In [1]:
import pandas as pd 
import numpy as np 
import gower
import kmedoids
from sklearn.metrics import silhouette_score

In [2]:
client_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\clients_final_df.csv"
client_df = pd.read_csv(client_path, sep=';')

In [20]:
# Būs vajadzīgas kolonnas: age_group, gender, region, debt_amount, days_from_delay_to_cession, number_of_cases_per_ssn
clustering_df = client_df[['Age_group', 'Reģions', 'Principal', 'debt_age_at_purchase_days', 'number_of_cases_per_ssn', 'legal_flag']].copy()
# Izņemam rindas, kur age_group is NaN, jo tās nevar izmantot klasterizācijā:
clustering_df = clustering_df.dropna(subset=['Age_group'])

In [21]:
# Standartizējam kolonnu nosaukumus, lai tie būtu vieglāk lietojami:
clustering_df.columns = ['age_group', 'region', 'debt_amount', 'days_from_delay_to_cession', 'number_of_cases_per_ssn', 'legal_flag']

1. Create debt_age_group

In [22]:
def create_debt_age_group(days):
    if pd.isna(days):
        return "UNKNOWN"
    elif days <= 90:
        return "0–90 days"
    elif days <= 180:
        return "91–180 days"
    elif days <= 365:
        return "181–365 days"
    elif days <= 730:
        return "1–2 years"
    elif days <= 1095:
        return "2–3 years"
    else:
        return "3+ years"


clustering_df["debt_age_group"] = clustering_df["days_from_delay_to_cession"].apply(create_debt_age_group)


# Pārveidojam legal_flag kolonnu uz bināro formātu, kur "legal" = 1 un "non-legal" = 0:
clustering_df["legal_flag_binary"] = (
    clustering_df["legal_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "legal": 1,
        "non-legal": 0
    })
)

# Izņemam kolonnas debt_age_at_purchase_days un legal_flag, jo tās vairs nav nepieciešamas:
clustering_df = clustering_df.drop(columns=['days_from_delay_to_cession', 'legal_flag'])

2. Prepare clustering dataset

In [23]:
cluster_features = [
    "age_group",
    "region",
    "debt_age_group",
    "debt_amount",
    "number_of_cases_per_ssn"
]

target_col = "legal_flag_binary"

cluster_df = clustering_df[cluster_features + [target_col]].copy()

3. Handle missing values

In [24]:
cluster_features = [
    "age_group",
    "region",
    "debt_amount",
    "debt_age_group",
    "number_of_cases_per_ssn"
]

cluster_df = clustering_df[cluster_features + [target_col]].copy()

categorical_cols = [
    "age_group",
    "region",
    "debt_age_group"
]

numeric_cols = [
    "debt_amount",
    "number_of_cases_per_ssn"
]

for col in categorical_cols:
    cluster_df[col] = cluster_df[col].astype("object").fillna("UNKNOWN")

for col in numeric_cols:
    cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce")
    cluster_df[col] = cluster_df[col].fillna(cluster_df[col].median())



4. Transform skewed numeric variables

In [25]:
cluster_df["debt_amount_log"] = np.log1p(cluster_df["debt_amount"])
cluster_df["number_of_cases_per_ssn_log"] = np.log1p(cluster_df["number_of_cases_per_ssn"])

5. Final features used for clustering

In [26]:
features_for_clustering = [
    "age_group",
    "region",
    "debt_age_group",
    "debt_amount_log",
    "number_of_cases_per_ssn_log"
]

X = cluster_df[features_for_clustering].copy()

6. Calculate Gower distance matrix

In [27]:
gower_dist = gower.gower_matrix(X)

distance_matrix = np.asarray(gower_dist, dtype=np.float64)

distance_matrix.shape

(14209, 14209)

In [28]:
results = []

for k in range(3, 8):
    pam_result = kmedoids.pam(
        distance_matrix,
        k,
        random_state=42
    )
    
    labels = pam_result.labels
    
    sil = silhouette_score(
        distance_matrix,
        labels,
        metric="precomputed"
    )
    
    results.append({
        "k": k,
        "silhouette": sil
    })

silhouette_results = (
    pd.DataFrame(results)
    .sort_values("silhouette", ascending=False)
)

silhouette_results

,k,silhouette
2,5,0.198932
3,6,0.196548
4,7,0.190551
1,4,0.166352
0,3,0.162804


8. Fit final PAM model

In [29]:
final_k = 5

pam_result = kmedoids.pam(
    distance_matrix,
    final_k,
    random_state=42
)

cluster_df["cluster"] = pam_result.labels + 1

In [30]:
clustering_df["cluster"] = cluster_df["cluster"]

9. Create cluster summary with legal rate

In [31]:
cluster_summary = (
    cluster_df
    .groupby("cluster")
    .agg(
        cases=(target_col, "size"),
        legal_rate=(target_col, "mean"),
        avg_debt_amount=("debt_amount", "mean"),
        median_debt_amount=("debt_amount", "median"),
        avg_number_of_cases_per_ssn=("number_of_cases_per_ssn", "mean"),
        median_number_of_cases_per_ssn=("number_of_cases_per_ssn", "median")
    )
    .reset_index()
    .sort_values("legal_rate", ascending=False)
)

cluster_summary

,cluster,cases,legal_rate,avg_debt_amount,median_debt_amount,avg_number_of_cases_per_ssn,median_number_of_cases_per_ssn
4,5,2623,0.732368,970.001041,707.720,1.834922,1.0
0,1,3287,0.704594,1608.480231,1000.000,1.726194,1.0
3,4,3223,0.646292,976.229960,691.690,2.320509,2.0
2,3,2308,0.590121,1203.032400,496.910,1.476603,1.0
1,2,2768,0.523121,436.565119,74.575,1.237355,1.0


10. Add most common categorical values per cluster

In [33]:
def most_common_value(series):
    counts = series.value_counts(dropna=False)
    return counts.index[0]


categorical_summary = (
    cluster_df
    .groupby("cluster")
    .agg(
        most_common_age_group=("age_group", most_common_value),
        most_common_region=("region", most_common_value),
        most_common_debt_age_group=("debt_age_group", most_common_value)
    )
    .reset_index()
)

cluster_profile = cluster_summary.merge(
    categorical_summary,
    on="cluster",
    how="left"
)

cluster_profile

,cluster,cases,legal_rate,avg_debt_amount,median_debt_amount,avg_number_of_cases_per_ssn,median_number_of_cases_per_ssn,most_common_age_group,most_common_region,most_common_debt_age_group
0,5,2623,0.732368,970.001041,707.720,1.834922,1.0,<30,Rīga,91–180 days
1,1,3287,0.704594,1608.480231,1000.000,1.726194,1.0,31-50,Rīga,91–180 days
2,4,3223,0.646292,976.229960,691.690,2.320509,2.0,31-50,Vidzemes reģions,0–90 days
3,3,2308,0.590121,1203.032400,496.910,1.476603,1.0,51-70,Rīga,91–180 days
4,2,2768,0.523121,436.565119,74.575,1.237355,1.0,31-50,Rīga,UNKNOWN


11. Distribution tables for interpretation

In [16]:
# Debt age distribution by cluster
debt_age_distribution = (
    pd.crosstab(
        cluster_df["cluster"],
        cluster_df["debt_age_group"],
        normalize="index"
    )
    .round(3)
)

debt_age_distribution

debt_age_group,0–90 days,181–365 days,1–2 years,2–3 years,3+ years,91–180 days,UNKNOWN
cluster,,,,,,,
1,0.059,0.066,0.054,0.029,0.079,0.525,0.188
2,0.159,0.089,0.043,0.021,0.058,0.581,0.048
3,0.185,0.102,0.010,0.014,0.041,0.584,0.065
4,0.148,0.064,0.096,0.046,0.123,0.034,0.488
5,0.722,0.086,0.012,0.007,0.036,0.052,0.084


In [17]:
# Age group distribution by cluster
age_distribution = (
    pd.crosstab(
        cluster_df["cluster"],
        cluster_df["age_group"],
        normalize="index"
    )
    .round(3)
)

age_distribution

age_group,31-50,51-70,71-90,90+,<30,legal/new SSN
cluster,,,,,,
1,0.783,0.115,0.039,0.002,0.060,0.001
2,0.678,0.102,0.052,0.000,0.165,0.002
3,0.060,0.109,0.021,0.000,0.808,0.001
4,0.113,0.626,0.102,0.018,0.108,0.032
5,0.692,0.107,0.032,0.000,0.167,0.001


In [18]:
# Region distribution by cluster
region_distribution = (
    pd.crosstab(
        cluster_df["cluster"],
        cluster_df["region"],
        normalize="index"
    )
    .round(3)
)

region_distribution

region,Daugavpils,Jelgava,Jēkabpils,Jūrmala,Kurzemes reģions,Latgales reģions,Liepāja,Ogre,Rēzekne,Rīga,Rīgas reģions,UNKNOWN,Valmiera,Ventspils,Vidzemes reģions,Zemgales reģions,nav indeksa / ārzemnieks
cluster,,,,,,,,,,,,,,,,,
1,0.026,0.026,0.015,0.020,0.064,0.063,0.072,0.016,0.011,0.397,0.080,0.034,0.012,0.018,0.043,0.103,0.001
2,0.028,0.033,0.018,0.017,0.092,0.066,0.060,0.012,0.013,0.332,0.078,0.025,0.016,0.015,0.100,0.096,0.000
3,0.022,0.033,0.020,0.018,0.335,0.064,0.060,0.010,0.009,0.086,0.091,0.030,0.011,0.023,0.082,0.104,0.002
4,0.024,0.020,0.011,0.021,0.070,0.051,0.117,0.010,0.012,0.395,0.059,0.024,0.010,0.016,0.080,0.079,0.001
5,0.023,0.025,0.013,0.016,0.075,0.074,0.043,0.012,0.013,0.090,0.087,0.018,0.014,0.018,0.357,0.123,0.000


12. Check legal rate by debt age group

In [19]:
legal_by_debt_age_group = (
    cluster_df
    .groupby("debt_age_group")
    .agg(
        cases=(target_col, "size"),
        legal_rate=(target_col, "mean"),
        avg_debt_amount=("debt_amount", "mean"),
        median_debt_amount=("debt_amount", "median")
    )
    .reset_index()
    .sort_values("legal_rate", ascending=False)
)

legal_by_debt_age_group

,debt_age_group,cases,legal_rate,avg_debt_amount,median_debt_amount
5,91–180 days,5356,0.699403,1349.484274,875.905
0,0–90 days,3472,0.672235,933.312716,713.500
1,181–365 days,1149,0.615318,1072.424909,735.350
6,UNKNOWN,2341,0.576249,917.590500,142.980
3,2–3 years,332,0.554217,867.297259,99.765
4,3+ years,950,0.538947,186.204326,96.000
2,1–2 years,609,0.489327,1066.265255,122.410
